## RAG pipeline

#### Data ingestion + chunking

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pandas as pd

import os
from pathlib import Path

# The below code is used to access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings

In [3]:
print(os.getcwd())

c:\Users\USER\Documents\AI projects\DenseLess\app\agent


In [4]:
docs = process_and_load_file("../pdfs/Interconnection Structures.pdf")
print(docs)

PDF page count: 10
Processing: Interconnection Structures.pdf


INFO: split_pdf event=plan_created operation_id=dacbc8ff-d845-431f-8b46-d8df884ce40a filename=Interconnection Structures.pdf strategy=hi_res page_range=1-10 page_count=10 split_size=2 chunk_count=5 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset
INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=dacbc8ff-d845-431f-8b46-d8df884ce40a chunk_count=5 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https:/

  → Parsed into 113 document(s)
[Document(metadata={'section': 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK 2)', 'category': 'Title', 'page_number': 1, 'element_id': 'fc8d39a607237108cc91e2af962c85be', 'filename': 'Interconnection Structures.pdf'}, page_content='COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK 2)'), Document(metadata={'section': 'Interconnection Structures:', 'category': 'Title', 'page_number': 1, 'element_id': '78c7c84181e4734980a9b02f62fb3c9e', 'filename': 'Interconnection Structures.pdf'}, page_content='Interconnection Structures:'), Document(metadata={'section': 'Interconnection Structures:', 'category': 'ListItem', 'page_number': 1, 'element_id': '1cb8263010cac730f3abb32370c847ac', 'filename': 'Interconnection Structures.pdf'}, page_content='• Bus structures: Single bus, Multiple bus'), Document(metadata={'section': 'Interconnection Structures:', 'category': 'ListItem', 'page_number': 1, 'element_id': 'eaabff3113b41dbce12353569adcc14d', 'filename': 'Inter

In [5]:
pd.set_option('display.max_rows', None)

df = pd.DataFrame([
    {**doc.metadata, "content_preview": doc.page_content[:300], "character count": len(doc.page_content)}
    for doc in docs
])
df

,section,category,page_number,element_id,filename,content_preview,character count
0,COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (...,Title,1,fc8d39a607237108cc91e2af962c85be,Interconnection Structures.pdf,COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (...,53
1,Interconnection Structures:,Title,1,78c7c84181e4734980a9b02f62fb3c9e,Interconnection Structures.pdf,Interconnection Structures:,27
2,Interconnection Structures:,ListItem,1,1cb8263010cac730f3abb32370c847ac,Interconnection Structures.pdf,"• Bus structures: Single bus, Multiple bus",42
3,Interconnection Structures:,ListItem,1,eaabff3113b41dbce12353569adcc14d,Interconnection Structures.pdf,"• Instruction set Architecture: CISC, RISC",42
4,Interconnection Structures:,ListItem,1,7001f1a816ea15e1c3df74115c8dcfb0,Interconnection Structures.pdf,• Instruction and instruction sequencing:,41
5,Interconnection Structures:,ListItem,1,15a482b33c2c80a1e7d7a1339ce9ec7e,Interconnection Structures.pdf,"• Register transfer notation, Instruction types.",48
6,Interconnection structures,Title,1,9a2a4865526877ef04d568afbaf1e904,Interconnection Structures.pdf,Interconnection structures,26
7,Interconnection structures,ListItem,1,93fde35d683dc0b575c175f529a80ed1,Interconnection Structures.pdf,• Computers has three main or basic modules: P...,70
8,Interconnection structures,ListItem,1,d760de483bd8bbe1ed2e628e35ce7625,Interconnection Structures.pdf,o The three basic modules must communicate wit...,159
9,Interconnection structures,NarrativeText,1,3fd586755419dc086de66bfc7c99eaf2,Interconnection Structures.pdf,The collection of paths connecting the various...,186


In [6]:
pd.set_option('display.max_rows', 40)
print(df)

                                               section       category  \
0    COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (...          Title   
1                          Interconnection Structures:          Title   
2                          Interconnection Structures:       ListItem   
3                          Interconnection Structures:       ListItem   
4                          Interconnection Structures:       ListItem   
..                                                 ...            ...   
108                             INSTRUCTION SEQUENCING  NarrativeText   
109                                         References  NarrativeText   
110                                         References  NarrativeText   
111                                         References  NarrativeText   
112                                         References  NarrativeText   

     page_number                        element_id  \
0              1  fc8d39a607237108cc91e2af962c85be   
1              

In [7]:
len(docs[11].page_content)

61

#### Embeddings

In [8]:
embedder = get_embedding_model()
vectors = generate_embeddings(docs, embedder)
print(vectors)

Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


INFO: Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/confi

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree

  → Embedding model loaded successfully. Embedding dimension: 384
Generating embeddings for 113 document chunk(s)...


Embedding documents: 100%|██████████| 4/4 [00:01<00:00,  2.55it/s]

  → Successfully generated 113 embedding vectors.
  → Embeddings shape: (113, 384)
[[-0.03307743 -0.02252311  0.00362147 ...  0.01305835  0.05501697
   0.04015127]
 [-0.00905513 -0.02675503  0.02120095 ...  0.0057753   0.0591531
   0.06242672]
 [ 0.00781565 -0.05606753  0.02815581 ... -0.00476676 -0.00990322
   0.06489182]
 ...
 [-0.00161077  0.02684236  0.02257675 ...  0.00300936  0.04581733
   0.02135081]
 [-0.02171915  0.01779508 -0.00443761 ...  0.00275593  0.04390103
  -0.01045641]
 [-0.01058074 -0.02497308 -0.00528192 ...  0.00974092  0.07157794
  -0.0341115 ]]


In [9]:
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma

In [10]:
# This is my id
store = get_vector_store(student_id="1019", embedder=embedder)

Vector store backend: 'chroma' | student: '1019'
Initialising Chroma store — student: '1019'
  → Collection : student_1019
  → Persist dir: ../chroma_db
  → Chroma store ready. 
Documents in collection: 0


In [13]:
add_documents_to_chroma(store, vectors, docs, "CSC 315", "Interconnection Structures", "Interconnection Structures")

  [vector_store] 0 existing chunk(s) found in collection.
  [vector_store] Adding 113 new chunk(s) to Chroma store...
  [vector_store] ✓ Successfully added 113 chunk(s).
  [vector_store] Total documents in collection: 113


### Retrieval pipeline

In [14]:
from app.agent.rag.retrieval.retriever import get_semantic_chunks, get_topic_chunks
from IPython.display import display, Markdown

In [15]:
# Get relevant chunks through Q&A
chunks = get_semantic_chunks(query="What is CISC", store=store, score_threshold=0.4, top_k=5, topic="Interconnection Structures", course="CSC 315")
chunks

[Retriever] Phase 1 — Semantic search (top_k=5, threshold=0.4): 'What is CISC'
Querying Chroma (top_k=5, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 5 chunk(s), 1 passed threshold.
[Retriever] → 1 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 1 section(s): {'Complex Instruction Set Architecture (CISC):'}
[Retriever]   → Section 'Complex Instruction Set Architecture (CISC):': 4 chunk(s) fetched.
[Retriever] → Expansion complete: 1 seed → 4 unique chunk(s) after deduplication.
[Retriever] → Final result: 4 chunk(s) returned to chain.


[Document(id='doc_d72d8e7a_39', metadata={'category': 'Title', 'content_length': 44, 'section': 'Complex Instruction Set Architecture (CISC):', 'page_number': 4, 'element_id': 'c7e6f4f3dd36572c3b52fddc8485d92e', 'course': 'CSC 315', 'source': 'CSC 315_Interconnection Structures_Interconnection Structures', 'filename': 'Interconnection Structures.pdf', 'topic': 'Interconnection Structures', 'doc_index': 39}, page_content='Complex Instruction Set Architecture (CISC):'),
 Document(id='doc_dd0bd7bc_40', metadata={'course': 'CSC 315', 'doc_index': 40, 'section': 'Complex Instruction Set Architecture (CISC):', 'topic': 'Interconnection Structures', 'filename': 'Interconnection Structures.pdf', 'page_number': 4, 'source': 'CSC 315_Interconnection Structures_Interconnection Structures', 'element_id': '0ef2dedacc16e3c7b47770518325a156', 'category': 'NarrativeText', 'content_length': 157}, page_content='Making hardware complex in order accommodate very complex low level language statements that 

In [16]:
texts = [doc.page_content for doc in chunks]
for i, text in enumerate(texts):
    display(Markdown(f"### Chunk {i+1}\n\n{text}\n"))

### Chunk 1

Complex Instruction Set Architecture (CISC):


### Chunk 2

Making hardware complex in order accommodate very complex low level language statements that a single instruction will do all loading, operation and storing.


### Chunk 3

The main idea is that a single instruction will do all loading, evaluating, and storing operations just like a multiplication command will do stuff like loading data, evaluating, and storing it, hence it’s complex.


### Chunk 4

Both approaches try to increase the CPU performance


In [19]:
# Get chunks for topic
chunks_topic = get_topic_chunks(store)
chunks_topic

[Retriever] Topic sweep — fetching all chunks from store...
[Retriever] → 113 chunk(s) across 17 section(s).
[Retriever] → Topic map built: 17 section(s), all chunks retained.


{'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK 2)': [Document(id='doc_384a2ce7_0', metadata={'course': 'CSC 315', 'category': 'Title', 'section': 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK 2)', 'element_id': 'fc8d39a607237108cc91e2af962c85be', 'doc_index': 0, 'page_number': 1, 'topic': 'Interconnection Structures', 'content_length': 53, 'source': 'CSC 315_Interconnection Structures_Interconnection Structures', 'filename': 'Interconnection Structures.pdf'}, page_content='COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK 2)')],
 'Interconnection Structures:': [Document(id='doc_c54a5b20_2', metadata={'category': 'ListItem', 'source': 'CSC 315_Interconnection Structures_Interconnection Structures', 'page_number': 1, 'section': 'Interconnection Structures:', 'course': 'CSC 315', 'element_id': '1cb8263010cac730f3abb32370c847ac', 'filename': 'Interconnection Structures.pdf', 'content_length': 42, 'doc_index': 2, 'topic': 'Interconnection Structures'}, page_content='• Bus struc

In [20]:
topic_chunk_result = [doc for doc_list in chunks_topic.values() for doc in doc_list]
topic_chunk_result
for i, doc in enumerate(topic_chunk_result):
    display(Markdown(f"\n\n{doc.page_content}\n"))



COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK 2)




• Bus structures: Single bus, Multiple bus




• Instruction and instruction sequencing:




• Instruction set Architecture: CISC, RISC




Interconnection Structures:


...



• The processor control circuits use information in PC to fetch & execute instructions one at a time in order of increasing address. This is called straight line sequencing.




https://gowsalyaa24.wordpress.com/2011/04/28/instructions-and-instruction-sequencing/




https://www.sciencedirect.com/topics/computer-science




https://www.geeksforgeeks.org/computer-organization-risc-and-cisc




William Stallings, Computer Organization and Architecture, 7th edition, Prentice Hall International, Inc., 2010.
